# Validation early stopping versus theta estimation

This notebook displays the second diagnostic experiment. The data are paired within each original Experiment 1 trajectory. Existing 15-checkpoint paths were replayed once with the same seeds to record all 1,000 epochs; stopping rules are applied retrospectively. The full available source has **10 repetitions per case**, not 100.

Run cells in order with the repository `.venv` kernel. By default, this notebook only reads completed results. Set `RUN_ANALYSIS = True` to regenerate tables and figures from completed dense trajectories without training. The README gives commands for recording dense paths from a new source run.

In [ ]:
import sys, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p/'dqAux.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the dplqr repository.')
EXPERIMENT_DIR = ROOT/'results/2026-09-08-early-stopping-vs-theta'
RUN_ANALYSIS = False
CASE_TO_DISPLAY = 1  # 1: linear; 2: additive; 3: deep
print('Interpreter:', sys.executable)
print('Result folder:', EXPERIMENT_DIR)

In [ ]:
if RUN_ANALYSIS:
    subprocess.run([sys.executable, str(EXPERIMENT_DIR/'scripts/run_diagnostic.py'),
                    '--stage', 'analyze', '--output-dir', str(EXPERIMENT_DIR)], check=True)
required = ['monte_carlo_summary_patience.csv', 'monte_carlo_summary_max_epochs.csv',
            'oracle_epoch_summary.csv', 'paired_max_epoch_comparisons.csv']
for filename in required:
    path = EXPERIMENT_DIR/filename
    if not path.is_file():
        raise FileNotFoundError(f'{path} is missing. Run the README smoke/all commands first.')
    frame = pd.read_csv(path)
    display(Markdown('### ' + filename))
    display(frame.loc[frame.case.eq(CASE_TO_DISPLAY)])

## Figures for the chosen case

A: coefficient RMSE versus patience. B: selected epoch versus patience. C: selected epoch versus the cap. D: prediction error versus coefficient error. E: validation-selected epoch versus the infeasible theta oracle.

The validation rule restores the earliest strict best checkpoint. Its criterion averages the two validation batch losses equally (128 and 72 observations); the ordinary empirical validation mean is also retained. Oracle epochs never affect this decision. These figures diagnose estimation; they do not measure confidence-interval coverage.

In [ ]:
figures = sorted((EXPERIMENT_DIR/'figures').glob(f'case_{CASE_TO_DISPLAY}_*.png'))
if not figures:
    raise FileNotFoundError('No figures found; run the analysis command from the README.')
for figure in figures:
    display(Image(filename=str(figure)))